# 批量处理 vertical_well_las_target 的 LAS

本笔记会对 data/vertical_well_las_target 下全部 LAS 文件执行连续相同值区间替换：

- 严格相等判定
- 连续长度 >= min_run_length 即替换为 anomaly_value
- 当前 anomaly_value 设置为 -99999.0
- 缺失值 (-999.0, -999.25, -99999, NaN) 不参与判定

处理结果会写到每个输入文件同目录下的 output 子目录，并在下方展示简洁报告。


In [ ]:
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "src").exists() and (p / "data").exists():
            return p
    raise RuntimeError("未找到仓库根目录（需要同时包含 src 和 data 目录）。")


cwd = Path.cwd().resolve()
ROOT = find_repo_root(cwd)
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from cup.well.raw import replace_constant_value_intervals_in_las

input_dir = ROOT / "data" / "vertical_well_las_target"
# Windows 大小写不敏感，按解析后的绝对路径去重，避免同一文件被重复处理。
las_candidates = [*input_dir.glob("*.las"), *input_dir.glob("*.Las"), *input_dir.glob("*.LAS")]
las_files = sorted({p.resolve(): p for p in las_candidates}.values(), key=lambda p: p.name.lower())

if not las_files:
    raise RuntimeError(f"目录下未找到 LAS 文件: {input_dir}")

min_run_length = 8
anomaly_value = -99999.0
write_fmt = "%.6f"

reports = []
for las_path in las_files:
    report = replace_constant_value_intervals_in_las(
        las_file_path=str(las_path),
        min_run_length=min_run_length,
        anomaly_value=anomaly_value,
        write_fmt=write_fmt,
    )
    reports.append(report)

summary_df = pd.DataFrame(
    {
        "file": [Path(r["input_file"]).name for r in reports],
        "output_file": [r["output_file"] for r in reports],
        "anomaly_value": [r["anomaly_value"] for r in reports],
        "write_fmt": [r["write_fmt"] for r in reports],
        "target_curve_count": [r["target_curve_count"] for r in reports],
        "curves_with_replacement": [r["curves_with_replacement"] for r in reports],
        "total_replaced_points": [r["total_replaced_points"] for r in reports],
    }
)

summary_df

In [ ]:
detail_rows = []
for report in reports:
    file_name = Path(report["input_file"]).name
    for item in report["curve_reports"]:
        detail_rows.append(
            {
                "file": file_name,
                "curve": item["curve"],
                "curve_index": item["curve_index"],
                "run_count": item["run_count"],
                "replaced_points": item["replaced_points"],
            }
        )

detail_df = pd.DataFrame(detail_rows)

if detail_df.empty:
    print("所有文件处理完成，但未发现达到阈值的连续相同值区间。")
else:
    detail_df = detail_df.sort_values(["file", "replaced_points"], ascending=[True, False]).reset_index(drop=True)
    detail_df